In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re
from wordcloud import STOPWORDS, WordCloud
from config import DATA_PROCESSED_PATH

In [ ]:
df = pd.read_csv(DATA_PROCESSED_PATH)
df = df.rename(columns={'message': 'text'})
df['message_length'] = df['text'].fillna('').apply(len)

label_count = df['label'].value_counts()

plt.figure(figsize=(10,4))
plt.pie(label_count, labels=['Ham', 'Spam'], autopct='%1.2f%%',colors = ['#ADCBD7','#7FA1C3'])
plt.title('Distribution of Ham versus Spam messages')
plt.show()

In [ ]:
def style_table(data_style):
    return( data_style.style
        .hide(axis='index')
        .set_table_styles([ #properties of column headers
            {'selector': 'th',
            'props':
                [('background-color', '#002D4C'),
                ('color','white'),
                ('border','1px solid #3b3b3b'),]}
    ])
        .set_properties(**{  #properties of each cell
            'background-color': '#ADC9E4',
            'color': 'black',
            'text-align': 'center',
            'border':'1px solid #3b3b3b'})

        .set_properties(subset=['Message length'],**{
            'background-color': '#E1EBEE'})

        .highlight_max(
            subset=['Ham messages','Spam messages'],
            axis = 0,
            props = 'background-color: #7FA1C3;')
)

bins = [0, 50, 100, 200, 300, 400, 500, 1000]
range_labels = ['1-50','51- 100','101-200','201-300','301-400','401-500','500+']

data = (df.groupby([pd.cut(df['message_length'],bins = bins, labels=range_labels),'label'])
        .size()
        .unstack(fill_value=0)
        .rename(columns={0: 'Ham messages', 1: 'Spam messages'}) 
        .reset_index()
        .rename(columns={'message_length': 'Message length'}))

style_table(data)

In [ ]:
def style_words(data_style):
    return(data_style.style
           .hide(axis='index')
           .set_properties(**{ #properties of each cell
                'font-size': '14px',
                'color': 'black',
                'text-align': 'center',
                'border':'1px solid #3b3b3b'})
           .set_properties(**{ #properties of the first two columns
                'background-color': '#ADC9E4'},subset=data_style.columns[0:2])
           .set_properties(**{ #properties of the last two columns
                'background-color': '#7FA1C3'},subset=data_style.columns[2:4])
           .set_table_styles([ #properties of column headers
                {'selector': 'th','props': [
                    ('background-color', '#002D4C'),
                    ('color','white'),
                    ('text-align','center'),
                    ('font-size','16px'),
                    ('border','1px solid #3b3b3b')]}
        ]))


def get_top_words(group, n = 10):
    clean_group = group.dropna().astype(str)
    full_text = " ".join(clean_group).lower()
    words = re.findall(r'\b\w{3,}\b', full_text) #Words that are less than 3 letters are ignored in the table

    filtered = [w for w in words if w not in stop_words]
    return pd.DataFrame(Counter(filtered).most_common(n), columns = ['word','count'])


stop_words = set(STOPWORDS)


spam_data = get_top_words(df[df['label'] == 1]['text']) 
ham_data = get_top_words(df[df['label'] == 0]['text']) 
top_words_df = pd.concat([spam_data,ham_data],axis=1)

top_words_df.columns = pd.MultiIndex.from_tuples([('SPAM','Word'),('SPAM','Frequency'),('HAM','Word'),('HAM','Frequency')])

style_words(top_words_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.subplots_adjust(wspace=0.1)

for ax, label, title in zip(axes, [1, 0], ['Spam', 'Ham']):
    words = get_top_words(df[df['label'] == label]['text'], n=150)
    wc = WordCloud(
        background_color='white',
        width=800,
        height=400,
        colormap='YlGnBu',
        max_words=150,
        contour_width=1,
    ).generate_from_frequencies(dict(words.values))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{title} Word Cloud', fontsize=14, fontweight='bold', pad = 10)

plt.tight_layout()
plt.show()

In [ ]:
spam_large = get_top_words(df[df['label'] == 1]['text'], n=150)
ham_large = get_top_words(df[df['label'] == 0]['text'], n=150)

common_words ={**dict(spam_large.values), **dict(ham_large.values)}

wc = WordCloud(
    background_color = 'white',
    width = 800,
    height = 400,
    colormap = 'YlGnBu',
    max_words = 150, #To display less words, change max_words parameter
    contour_width= 1,

).generate_from_frequencies(common_words)

plt.figure(figsize=(10,6))
plt.imshow(wc, interpolation='bilinear')
plt.title("Combined Word Cloud",fontsize=14, fontweight='bold', pad = 10)
plt.axis("off")
plt.show()
